# Cấp độ 2 — GRKT (KDD 2024) trên Kaggle GPU

Notebook **clone repo** [Knowledge-Tracing](https://github.com/linhnnh688/Knowledge-Tracing) rồi gọi hàm từ package `level2/`. Toàn bộ logic thực nghiệm nằm trong package; notebook chỉ điều phối — cùng kiến trúc với `kaggle_level1.ipynb`.

Mô hình chính: **GRKT — Knowledge Tracing với truy xuất trên đồ thị quan hệ** (Cui et al., KDD 2024), mã nguồn vendored trong thư mục `GRKT/`.

Phạm vi notebook = đúng yêu cầu Cấp độ 2: chạy GRKT trên **≥ 2 tập dữ liệu công khai** với cùng quy trình chia dữ liệu và cùng chỉ số, trực quan hoá mastery theo khái niệm/thời gian, đo ≥ 2 tiêu chí hợp lý của quỹ đạo. Ablation cấu trúc đồ thị và đa seed thuộc Cấp độ 1 (đã có kết quả) nên mặc định TẮT ở đây; calibration là phân tích nâng cao thứ hai, đề bài chỉ cần chọn **một** — đã chọn quỹ đạo mastery.

| Yêu cầu đề bài (Cấp độ 2) | Triển khai trong notebook |
|---|---|
| Mô hình graph-aware từ 2024+ | **GRKT**: 3 cơ chế *retrieval / strengthening / learning-forgetting* trên `rel_map` + `pre_map` |
| ≥ 2 tập dữ liệu công khai, cùng quy trình | `assist2009` (CSV chuẩn hoá của Cấp độ 1) và `junyi` (Kaggle) — cùng split theo người học, cùng chỉ số |
| Chống rò rỉ | dùng lại **đúng split seed 42 của Cấp độ 1**; `rel_map`/`pre_map` chỉ đếm từ tập train; in `leak_report.txt`; chuỗi GRKT được **đối chiếu từng người học** với bundle Cấp độ 1 |
| Trực quan hoá mastery theo thời gian | `mastery_<graph>.csv` = bảng dài m(u,c,t), nhiệt độ khái niệm × thời điểm, đường cong mastery |
| ≥ 2 tiêu chí hợp lý của quỹ đạo | **hướng lan truyền** (`consistency_spread`), **tính cục bộ** (`locality`), **suy giảm theo thời gian** (`forget_slope`) |
| So sánh cùng quy trình | bảng DKT / GKT-Transition / GKT-Zero / GRKT trên cùng split (đường nền **nạp checkpoint Cấp độ 1**, không train lại) |
| Ablation cấu trúc đồ thị | thuộc **Cấp độ 1** (đã có kết quả); GRKT hỗ trợ `train`/`dense`/`zero`/`shuffled` khi bật `RUN_GRAPH_ABLATIONS=True` |
| Phân tích nâng cao (đề bài chỉ cần **một**) | đã chọn **quỹ đạo mastery + tiêu chí hợp lý**; ECE/Brier có sẵn trong mã, bật `RUN_CALIBRATION=True` |
| Đa seed | DKT/GKT đã có Mean ± Std 3 seed ở Cấp độ 1 trên cùng split; `RUN_MULTI_SEED=True` nếu muốn thêm cho GRKT |

**Trước khi chạy**: tìm dataset **Junyi Academy** trong Kaggle Datasets, bấm **Agree/Subscribe** rồi `Add input` vào notebook. Package sẽ tự dò file trong `/kaggle/input` (không cần gõ đường dẫn). Nếu chưa attach, cell chuẩn hoá sẽ báo rõ là thiếu gì.

## 0. Clone repo và cấu hình môi trường

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/linhnnh688/Knowledge-Tracing.git'
KAGGLE_ROOT = Path('/kaggle/working')
IS_KAGGLE = KAGGLE_ROOT.exists()

if IS_KAGGLE:
    REPO = KAGGLE_ROOT / 'Knowledge-Tracing'
    if not (REPO / '.git').exists():
        if REPO.exists():
            shutil.rmtree(REPO, ignore_errors=True)
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    else:
        try:
            subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', 'main', '--depth', '1'])
            subprocess.check_call(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'])
        except Exception as e:
            print(f'Git sync error ({e}), re-cloning...')
            shutil.rmtree(REPO, ignore_errors=True)
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
else:
    REPO = Path.cwd() if (Path.cwd() / 'level2').exists() else Path.cwd() / 'Knowledge-Tracing'

os.chdir(REPO)

# REPO và REPO/GKT phải nằm trong sys.path: level1/ nạp pipeline của GKT theo kiểu module-top-level
for p in [REPO, REPO / 'GKT']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('Repo directory:', REPO.resolve())
print('Cấp độ 2 package:', (REPO / 'level2').resolve(), '| tồn tại:', (REPO / 'level2').is_dir())
print('GRKT vendored   :', (REPO / 'GRKT').resolve(), '| tồn tại:', (REPO / 'GRKT').is_dir())

In [ ]:
# Kaggle GPU image đã có sẵn PyTorch bản chạy được GPU + pandas/numpy/matplotlib/sklearn/tqdm.
# TUYỆT ĐỐI không `pip install torch` (sẽ tải bản CPU ~800MB và làm hỏng môi trường), và cũng
# không cố định phiên bản pandas/numpy: image hiện chạy Python 3.13 + pandas 3.x, code trong
# level1/level2 đã viết tương thích cả pandas 2 lẫn 3 (vẽ hình bằng matplotlib thuần).
import importlib.util
import subprocess
import sys

# tên module -> tên gói pip, chỉ cài những thứ thật sự thiếu
OPTIONAL = {"pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "tqdm": "tqdm"}
missing = [pkg for mod, pkg in OPTIONAL.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Cài bổ sung (chỉ các gói chưa có):", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Mọi thư viện đã có sẵn trong image — không cài gì thêm.")

import pandas as _pd, numpy as _np, matplotlib as _mpl
print("python", sys.version.split()[0], "| numpy", _np.__version__, "| pandas", _pd.__version__,
      "| matplotlib", _mpl.__version__)


### 0.2 Cấu hình thực nghiệm

* `QUICK_RUN = True` — dùng tập `assist2009_smoke` (15 người học, file smoke test của Cấp độ 1), 2 epoch, chỉ đồ thị `train`: kiểm tra mã trong ~3 phút. **Không dùng số liệu này cho báo cáo.**
* `QUICK_RUN = False` — `assist2009` đầy đủ + `junyi` (giới hạn số người học), GRKT với đồ thị `train`, quỹ đạo mastery + tiêu chí hợp lý: **cấu hình cho báo cáo**. Ba công tắc ngoài phạm vi yêu cầu Cấp độ 2 (`RUN_GRAPH_ABLATIONS`, `RUN_CALIBRATION`, `RUN_MULTI_SEED`) đều mặc định `False`; bật lên nếu session còn dư thời lượng GPU.
* `MAX_SEQ_LEN = 200` ở chế độ báo cáo **để khớp đúng Cấp độ 1** (Cấp độ 1 train DKT/GKT với `MAX_SEQ_LEN=200`). Nhờ vậy checkpoint DKT nạp lại được đánh giá trên đúng cửa sổ nó đã học, và mục 4 đối chiếu AUC với 0.7866 là so sánh trực tiếp. Nếu hạ xuống 80 thì đường nền bị cắt ngắn hơn lúc train → AUC lệch là **lệch do cửa sổ, không phải do split**, và bước đối chiếu sẽ báo ⚠.
* GRKT **nặng hơn GKT Cấp độ 1 nhiều lần** (mỗi bước thời gian chạy vài chục phép nhân ma trận trên toàn đồ thị) và MỖI biến thể đồ thị là một lần train riêng, nên `RUN_GRAPH_ABLATIONS=True` tốn gấp ~4 thời gian. Đó là lý do mặc định chỉ train đồ thị `train`. Nếu vẫn vượt thời lượng Kaggle: giảm `EPOCHS` (5 → 3) trước, chỉ giảm `MAX_SEQ_LEN` khi thật sự bí.
* `RESUME_GRKT_CKPT = True` — bỏ qua bước train GRKT của các biến thể đã có checkpoint `GRKT-<graph>.pt` trong `SAVE_DIR` (dùng để cứu một lần chạy đã train xong nhưng chết ở bước vẽ hình). Mặc định `False`.
* `MAX_USERS` chỉ ảnh hưởng tới tập phải chuẩn hoá (Junyi). `assist2009` dùng thẳng CSV đã commit nên luôn giữ đủ người học và giữ đúng split của Cấp độ 1.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from level2.dataset_spec import ensure_canonical, get_spec, probe_columns, find_kaggle_input
from level2.experiment import run_level2

print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QUICK_RUN = True

DATASETS = ["assist2009_smoke"] if QUICK_RUN else ["assist2009", "junyi"]
EPOCHS = 2 if QUICK_RUN else 5
BATCH_SIZE = 16 if QUICK_RUN else 64
# 200 = đúng cấu hình Cấp độ 1; đổi số khác thì mục 4 (đối chiếu DKT 0.7866) sẽ lệch.
MAX_SEQ_LEN = 50 if QUICK_RUN else 200
MAX_USERS = 300 if QUICK_RUN else 4000          # chỉ áp dụng cho tập cần chuẩn hoá (junyi)

# Cấp độ 2 chỉ yêu cầu: tái lập GRKT trên >=2 tập công khai với CÙNG split + CÙNG chỉ số,
# trực quan hoá mastery theo khái niệm/theo thời gian, và đo >=2 tiêu chí hợp lý của quỹ đạo.
# Ba núm dưới đây ngoài phạm vi đó nên mặc định TẮT; bật lại khi session còn dư thời lượng GPU.
RUN_BASELINES = True                    # DKT / GKT-Transition / GKT-Zero: nạp checkpoint Cấp độ 1, không train lại
# Ablation cấu trúc đồ thị là yêu cầu CẤP 1 (đã có kết quả trong kaggle_level1.ipynb).
# Bật True ở đây = train GRKT 4 lần cho mỗi tập (tốn gấp ~4 thời gian), chỉ để so sánh quỹ đạo.
RUN_GRAPH_ABLATIONS = False
VARIANTS = ("train", "dense", "zero", "shuffled") if RUN_GRAPH_ABLATIONS else ("train",)
RECORD_TRAJECTORY = True                # Nhiệm vụ 2 (BẮT BUỘC): m(u,c,t) + tiêu chí hợp lý
# YEU_CAU.md mục 7: Cấp độ 2 chỉ cần chọn MỘT phân tích nâng cao -> đã chọn quỹ đạo mastery.
RUN_CALIBRATION = False                 # ECE / Brier vẫn chạy được khi đặt True
# 3 seed của DKT/GKT đã chạy ở Cấp độ 1 trên đúng split này; GRKT nặng hơn nhiều lần.
RUN_MULTI_SEED = False
SEEDS = [42, 123, 456]
# B4 sẽ NẠP GRKT-<graph>.pt có sẵn thay vì train lại khi đặt True. Dùng cho tình huống đã train
# xong hàng giờ trên Kaggle rồi cell chết ở bước vẽ hình/ghi sản phẩm (như lỗi markersize ở B7).
# TẮT mỗi khi đổi split_seed / variants / cold_thresh / graph_thresh / d_hidden so với lần chạy
# trước: load_state_dict chỉ phát hiện khác KIẾN TRÚC, không phát hiện khác đồ thị hay khác split.
RESUME_GRKT_CKPT = False

# Siêu tham số GRKT (mặc định theo bài báo; k_hidden nhỏ để bớt nặng trên T4)
D_HIDDEN, K_HIDDEN, K_HOP = 128, 16, 1
LR = 1e-3
PATIENCE = 2 if QUICK_RUN else 3
GRAPH_THRESH = 0.6      # ngưỡng nhị phân hoá rel/pre đếm từ train
# Ngưỡng lạnh: với tập smoke (15 người học) mọi cặp khái niệm đều hiếm, để 5 thì đồ thị train RỖNG
# và ablation vô nghĩa -> hạ xuống 1 để đồ thị train vẫn còn cạnh cho GRKT học.
COLD_THRESH = 1 if QUICK_RUN else 5
ALPHA = 0.01            # hệ số khoảng thời gian của GRKT
GAP_CAP_MIN = 240       # chặn Δt (phút) — xem level2/grkt_data._cumulative_times

USE_COMMITTED_CKPTS = True   # nạp DKT/GKT đã train ở Cấp độ 1 thay vì train lại
BASELINE_EPOCHS = 2 if QUICK_RUN else 20   # epoch khi PHẢI train lại đường nền (tập chưa có ckpt Cấp độ 1): 20 = đúng công thức Cấp độ 1
SPLIT_SEED = 42              # BẤT DI BẮC DỊCH: đúng seed split của Cấp độ 1

SAVE_DIR = Path("/kaggle/working/level2_out") if IS_KAGGLE else (REPO / "outputs" / "level2")
LEVEL1_CKPT_DIR = REPO / "outputs" / "level1"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config: datasets={DATASETS} epochs={EPOCHS} batch={BATCH_SIZE} max_seq_len={MAX_SEQ_LEN}")
print(f"        variants={VARIANTS} multiseed={RUN_MULTI_SEED} trajectory={RECORD_TRAJECTORY} calibration={RUN_CALIBRATION}")
print("Output dir :", SAVE_DIR)
print("Level-1 ckpts:", LEVEL1_CKPT_DIR, LEVEL1_CKPT_DIR.is_dir())

## 1. GRKT làm gì, và notebook này đã sửa gì so với bản gốc

**Cơ chế (mỗi bước tương tác i):**

1. *Knowledge retrieval*: trạng thái mastery của mọi khái niệm lan truyền trên `rel_map` (đồng xuất hiện) và `pre_map` (tiên quyết, kèm `sub_map = pre_mapᵀ`) → `m̃`; dự đoán `P(đúng) = σ(m̃(c_i) − δ(q_i))`.
2. *Memory strengthening*: làm đúng thì **cộng** `gain`, làm sai thì **trừ** `loss` vào trạng thái của khái niệm vừa làm (và lan toả theo cạnh đồ thị).
3. *Learning–forgetting*: giữa hai bước, `exp(−(n+1)·Δt·k)` quyết định khái niệm nào được học thêm và khái niệm nào phai đi theo **thời gian thực**.

**Toàn bộ chỉnh sửa so với mã upstream** (được liệt kê lại trong `KE_HOACH.md`):

| # | Chỗ sửa | Lý do |
|---|---|---|
| 1 | `GRKT/` thành Python package, `import utils/metric/BaseModel` → import tương đối | `GKT/` (Cấp độ 1) cũng có `utils.py`; nếu không sửa thì nạp nhầm module của nhau |
| 2 | Bỏ `GRKT/Runner.py`: tự vòng lặp epoch trong `level2/grkt_api.py` | Runner cắt chuỗi thành cửa sổ 100 và chia **k-fold trên cửa sổ** ⇒ cùng một người học nằm ở cả train lẫn test (phạm §6.4). Mỗi người học giờ là MỘT chuỗi, split lấy từ Cấp độ 1 |
| 3 | Dựng lại `rel_map`/`pre_map` từ **train only** (`level2/grkt_data.build_relevance_maps`) | File `assist09.bin` gốc đếm đồ thị trên toàn bộ dữ liệu ⇒ rò rỉ (phạm §6.2) |
| 4 | Trục thời gian = **phút cộng dồn, chặn trên 240 phút** | `exp(−(n+1)·Δt·k)` với Δt thô (order_id/µs) underflow về 0 ⇒ mô hình "quên sạch" sau 1 bước, AUC chỉ bằng tỉ lệ đúng |
| 5 | `args.thresh = 0.5` và ma trận đưa vào đã nhị phân | Nhánh `if thresh == 0` của upstream gán **toàn bộ** ma trận = 1, kể cả hàng/cột padding số 0 |
| 6 | Thêm hook `rec` trong `GRKT/model/GRKT.py::forward` (3 điểm ghi) | Cấp master `m_pre/m_tilde/m_strong` cho quỹ đạo mastery; `rec is None` khi huấn luyện ⇒ không đổi hành vi, không thêm chi phí |
| 7 | `GRKT/Training.py`: `eval('GRKT.GRKT')` → registry tường minh, chặn dataset theo allowlist | bỏ thực thi chuỗi thành mã (bảo mật) và chặn nạp `assist09.bin` rò rỉ |

**Không sửa**: kiến trúc mô hình, công thức loss (BCE), `get_feed_dict`/`feature2tensor` (đệm batch), định nghĩa AUC/ACC. Vì vậy số liệu GRKT ở đây so sánh trực tiếp được với DKT/GKT của Cấp độ 1.

## 2. EDA và kiểm tra cột dữ liệu (bắt buộc trước khi chuẩn hoá Junyi)

Cell dưới in ra **tên cột thật** của từng tập dữ liệu. `level2/dataset_spec.py` khớp tên cột theo một danh sách ứng viên; nếu Kaggle đổi tên cột, nó sẽ **dừng và in ra các cột có sẵn** thay vì đoán mò — khi đó chỉ cần bổ sung tên cột vào `JUNYI.columns` rồi chạy lại.

In [ ]:
for name in DATASETS:
    spec = get_spec(name)
    print("=" * 72)
    print(f"[{name}] {spec.note}")
    if name == "junyi":
        guess = find_kaggle_input(patterns=("junyi",))
        print("  file gốc dò được trong /kaggle/input:", guess)
        if guess is None:
            print("  ⚠ Chưa thấy dữ liệu Junyi. Hãy Agree/Subscribe dataset rồi Add input vào notebook.")
    canonical = ensure_canonical(spec, out_dir=SAVE_DIR / name, max_users=MAX_USERS, verbose=True)
    print("  CSV chuẩn hoá:", canonical)
    print("  cột chuẩn hoá:", probe_columns(canonical))

    eda = pd.read_csv(canonical)
    print(f"  {len(eda):,} tương tác | {eda['user_id'].nunique():,} người học | "
          f"{eda['skill_id'].nunique():,} khái niệm | {eda['problem_id'].nunique():,} câu hỏi | "
          f"tỉ lệ đúng {eda['correct'].mean():.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    # Vẽ bằng matplotlib thuần: pandas 3 đã bỏ accessor .hist()/.barh() trực tiếp trên Series
    axes[0].hist(eda.groupby("user_id").size().to_numpy(), bins=40, edgecolor="black", alpha=0.7)
    axes[0].set_title(f"{name}: số tương tác mỗi người học")
    axes[0].set_xlabel("số lượt")
    top_skill = eda["skill_id"].value_counts().head(20).iloc[::-1]
    axes[1].barh([str(x) for x in top_skill.index], top_skill.to_numpy(), color="#4878a6")
    axes[1].set_title(f"{name}: 20 khái niệm thường gặp nhất")
    axes[1].set_xlabel("số lượt")
    fig.tight_layout()
    plt.show()

## 3. Chuỗi thực nghiệm Cấp độ 2

`run_level2(...)` gọi `run_dataset(...)` cho từng tập dữ liệu. Với cấu hình mặc định (chỉ chạy đúng những gì Cấp độ 2 yêu cầu), **B3/B4 chỉ sinh và train đồ thị `train`**, còn **B7 (calibration)** và **B8 (đa seed)** bị bỏ qua — bật `RUN_GRAPH_ABLATIONS` / `RUN_CALIBRATION` / `RUN_MULTI_SEED` nếu muốn chạy chúng:

| Bước | Nội dung | Sản phẩm |
|---|---|---|
| **B1** | chuẩn hoá CSV + split theo người học (seed 42, dùng lại pipeline Cấp độ 1) | `split.json` |
| **B2** | dựng chuỗi GRKT từ cùng file và **đối chiếu từng người học** với bundle Cấp độ 1 (khớp thì mới chạy tiếp) | log |
| **B3** | đếm `rel_map`/`pre_map` **chỉ từ train** (+ sinh dense/zero/shuffled nếu bật ablation) | `graphs_train_only.npz`, `leak_report.txt` |
| **B4** | huấn luyện GRKT cho từng biến thể được bật, early stop theo **val AUC toàn cục** | `GRKT-<graph>.pt` |
| **B5** | đường nền DKT / GKT-Transition / GKT-Zero trên cùng split (nạp checkpoint Cấp độ 1 nếu có) | `level2_metrics.csv` |
| **B6** | ghi quỹ đạo m(u,c,t) + 3 tiêu chí hợp lý | `traj_<graph>.npz`, `mastery_<graph>.csv`, `heatmap/curves`, `trajectory_metrics.csv` |
| **B7** | *(tuỳ chọn, mặc định tắt)* ECE / Brier / reliability cho mọi mô hình | `calibration.csv`, `reliability.png` |
| **B8** | *(tuỳ chọn, mặc định tắt — đa seed đã có ở Cấp độ 1)* split cố định, chỉ đổi khởi tạo + thứ tự batch | `multiseed_metrics.csv` |

In [ ]:
result = run_level2(
    DATASETS,
    save_root=SAVE_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    lr=LR,
    patience=PATIENCE,
    d_hidden=D_HIDDEN,
    k_hidden=K_HIDDEN,
    k_hop=K_HOP,
    alpha=ALPHA,
    graph_thresh=GRAPH_THRESH,
    cold_thresh=COLD_THRESH,
    gap_cap_min=GAP_CAP_MIN,
    max_users=MAX_USERS,
    variants=VARIANTS,
    split_seed=SPLIT_SEED,
    run_baselines=RUN_BASELINES,
    baseline_ckpt_dir=(LEVEL1_CKPT_DIR if USE_COMMITTED_CKPTS else None),
    baseline_epochs=BASELINE_EPOCHS,
    allow_retrain_baselines=True,
    record_trajectory=RECORD_TRAJECTORY,
    run_calibration=RUN_CALIBRATION,
    run_multi_seed=RUN_MULTI_SEED,
    seeds=SEEDS,
    resume_grkt_ckpt=RESUME_GRKT_CKPT,
    device=device,
)
combined = result["combined"]
print("\nSố dòng trong bảng nối:", len(combined))
combined

## 4. Bảng so sánh DKT / GKT / GRKT và kiểm tra đối chiếu split

In [ ]:
display(combined[["dataset", "model", "graph", "auc", "acc", "loss", "val_auc", "n_test"]])

# ---------------------------------------------------------------- kiểm tra đối chiếu split
# Đường nền DKT/GKT ở đây KHÔNG được train lại: checkpoint lấy từ outputs/level1/. Nếu Cấp độ 2
# đang thật sự dùng lại đúng split của Cấp độ 1 (cùng file, cùng seed 42, cùng max_seq_len), AUC
# đánh giá lại phải khớp số đã công bố trong vòng vài chữ số thập phân. Đây là bằng chứng thực
# nghiệm của yêu cầu "một split duy nhất cho mọi mô hình" (YEU_CAU.md §6.3).
LEVEL1_MAX_SEQ_LEN = 200
ref_file = LEVEL1_CKPT_DIR / "level1_metrics.csv"
ref = None
if ref_file.exists():
    ref = pd.read_csv(ref_file).rename(columns={"Mô hình": "model", "Test AUC": "ref_auc"})
    if MAX_SEQ_LEN != LEVEL1_MAX_SEQ_LEN:
        print(f"[cảnh báo] MAX_SEQ_LEN={MAX_SEQ_LEN} khác Cấp độ 1 ({LEVEL1_MAX_SEQ_LEN}): "
              "chuỗi eval bị cắt ngắn hơn lúc train checkpoint, lệch AUC là do cửa sổ cắt chuỗi. "
              "Đặt MAX_SEQ_LEN=200 nếu muốn đối chiếu khớp chính xác với Cấp độ 1.")
    tol = 0.02 if MAX_SEQ_LEN != LEVEL1_MAX_SEQ_LEN else 0.005
    for name in ["DKT", "GKT-Transition", "GKT-Zero"]:
        row = combined[(combined["dataset"] == "assist2009") & (combined["model"] == name)]
        rr = ref[ref["model"] == name]
        if len(row) and len(rr):
            a2, a1 = float(row["auc"].iloc[0]), float(rr["ref_auc"].iloc[0])
            flag = "OK (cùng split)" if abs(a2 - a1) <= tol else f"lệch > {tol} — kiểm tra lại split và max_seq_len"
            print(f"{name:16s} | Cấp độ 1 = {a1:.4f} | Cấp độ 2 = {a2:.4f} | Δ={a2 - a1:+.4f}  {flag}")
else:
    print("Không có outputs/level1/level1_metrics.csv — bỏ qua bước đối chiếu.")

if "assist2009" not in set(combined["dataset"]):
    print("[đối chiếu] Lần chạy này không có assist2009 (chế độ smoke dùng assist2009_smoke, "
          "hoặc chỉ chạy junyi) nên KHÔNG kiểm tra được AUC đường nền với Cấp độ 1. "
          "Bắt buộc chạy lại với DATASETS có 'assist2009' trước khi đưa số liệu vào báo cáo.")

fig, ax = plt.subplots(figsize=(9, 4))
sub = combined[combined["dataset"] == combined["dataset"].iloc[0]]
# .tolist() để so sánh bằng str thông thường, không phụ thuộc dtype của cột
graphs = sub["graph"].tolist()
bars = ax.bar(sub["model"], sub["auc"], color=["#9aa5b1" if g == "N/A" else "#4878a6" for g in graphs])
ax.set_ylabel("Test AUC")
ax.set_title(f"Test AUC trên {sub['dataset'].iloc[0]} — cùng một split, cùng một chỉ số")
ax.set_ylim(min(0.5, float(sub["auc"].min()) - 0.05), 0.85)
axhline_val = float(ref["ref_auc"][ref["model"] == "DKT"].iloc[0]) if ref is not None else 0.7866
ax.axhline(axhline_val, ls="--", lw=1, color="#c0392b", label=f"DKT Cấp độ 1 = {axhline_val:.4f}")
for b, v in zip(bars, sub["auc"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.004, f"{v:.4f}", ha="center", fontsize=8)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 5. Quỹ đạo mastery m(u, c, t) và các tiêu chí hợp lý

**Định nghĩa ba mốc đo trong một bước** (xem `level2/trajectory.py`):

* `m_pre[i]` — mastery của mọi khái niệm **trước** cập nhật ở bước i: đúng là trạng thái mô hình đã dùng để dự đoán.
* `m̃[i]` — mastery **sau giai đoạn truy xuất** (đã lan truyền trên cạnh đồ thị): khác `m_pre[i]` đúng ở phần lan truyền.
* `m_strong[i]` — mastery **sau giai đoạn củng cố** (đúng → cộng gain, sai → trừ loss).

Suy ra mỗi bước tách được hai cơ chế: `Δm_củng_cố = m_strong[i] − m_pre[i]` (ảnh hưởng của việc làm đúng/sai) và `Δm_học_quên = m_pre[i+1] − m_strong[i]` (ảnh hưởng của học + quên theo thời gian).

**Bốn tiêu chí hợp lý** (đề bài yêu cầu ≥ 2; cả bốn đều đo được trên đồ thị `train` nên cấu hình mặc định đã đủ):

1. `consistency_strengthen` — tại khái niệm vừa làm: đúng thì mastery phải tăng, sai thì phải giảm. Bất đẳng thức này đúng *theo cấu trúc* của GRKT, nên nó là **kiểm chứng cài đặt**: nếu ≠ 1 thì hook ghi quỹ đạo hoặc tiền xử lý đã sai.
2. `consistency_spread` — tại các khái niệm **láng giềng 1-hop** của khái niệm vừa làm (không kể nó): một lần làm bài có kéo theo khái niệm liên quan hay không. Đây là tiêu chí **có thể sai thật**, không chỉ là kiểm chứng cài đặt.
3. `locality` — tỉ lệ lượng thay đổi mastery nằm trong {đang làm ∪ láng giềng}. Cao = mô hình không "làm bẩn" những khái niệm không liên quan.
4. `forget_slope` — hồi quy Δm giữa hai bước theo Δt tại các khái niệm không bị chạm; kỳ vọng ≤ 0. Độ dốc dương nghĩa là mô hình cho rằng "không làm gì" cũng giỏi thêm.

Các câu so sánh chéo (`train` phải khác `shuffled`/`dense`/`zero`, `zero` có locality = 1.0 tầm thường, `|Δm| retrieval` của `zero` phải ~0) **chỉ xuất hiện khi bật `RUN_GRAPH_ABLATIONS=True`**, vì khi đó mỗi biến thể là một lần train riêng. Với cấu hình mặc định, bảng chỉ có dòng `train` — vẫn thoả yêu cầu "đo ≥ 2 tiêu chí hợp lý" của đề bài và không có cột nào để so sánh chéo.

In [ ]:
for name, run in result["runs"].items():
    tt = run.get("trajectory_table")
    if tt is None or len(run.get("trajectories", {})) == 0:
        print(f"[{name}] RECORD_TRAJECTORY = False, bỏ qua.")
        continue
    kind0 = sorted(run["trajectories"])[0]
    n_users = len(run["trajectories"][kind0]["users"])
    print("=" * 72)
    print(f"[{name}] TIÊU CHÍ HỢP LÝ CỦA QUỸ ĐẠO (đo trên {n_users} người học thuộc tập test)")
    print("=" * 72)
    display(tt)
    print("Giải thích: spread = None nghĩa là biến thể đó KHÔNG lan truyền sang láng giềng "
          "(đồ thị zero), không phải 'tệ' — hãy đọc cùng cột locality.\n")

### 5.1 Thay đổi mastery đến từ đâu (phân tách ba giai đoạn của GRKT)

GRKT cập nhật trạng thái theo ba bước: **retrieval** (đọc trạng thái các đỉnh kề sang khái
niệm đang làm), **strengthening** (tăng/giảm độ thành thạo theo kết quả vừa làm) rồi
**learn/forget** (lan truyền phần tăng đó sang láng giềng và suy giảm theo thời gian). Bảng
dưới đo `|Δm|` trung bình của từng giai đoạn trên cùng các người học, cho mỗi biến thể đồ thị
được bật. Với cấu hình mặc định chỉ có dòng `train`; ba gạch đầu dòng so sánh dưới đây cần
`RUN_GRAPH_ABLATIONS=True` (mỗi biến thể là một lần train riêng):

* `mean_abs_delta_retrieval` của biến thể `zero` **phải xấp xỉ 0** — không có cạnh thì không
  có gì để đọc. Nếu nó > 0 thì đồ thị `zero` đang không thật sự rỗng (lỗi tiền xử lý).
* `mean_abs_delta_strengthen` gần như không đổi giữa các biến thể — giai đoạn này chỉ nhìn kết
  quả làm bài của chính khái niệm đang trả lời.
* `mean_abs_delta_learn_forget` khác 0 ở MỌI biến thể (suy giảm theo thời gian là cơ chế nội
  tại). Tác động của đồ thị nằm ở CHÊNH LỆNH giữa `train`/`dense` và `zero`, không phải ở giá
  trị tuyệt đối.

In [ ]:
from level2.trajectory import stage_attribution

rows = []
for name, run in result["runs"].items():
    for kind, traj in sorted(run.get("trajectories", {}).items()):
        st = stage_attribution(traj)
        st.update({"dataset": name, "graph": kind})
        rows.append(st)
if rows:
    stage_df = pd.DataFrame(rows)
    cols = ["dataset", "graph"] + [c for c in stage_df.columns if c not in ("dataset", "graph")]
    display(stage_df[cols])
    zero = stage_df[stage_df["graph"] == "zero"]
    if len(zero):
        z = float(zero["mean_abs_delta_retrieval"].iloc[0])
        print(f"kiểm tra đồ thị rỗng: |Δm| retrieval của biến thể zero = {z:.4f} "
              + "(phải ~0; nếu > 0 thì graph_variant('zero') chưa thật sự không cạnh)")
    print("Với các biến thể có cạnh, so mean_abs_delta_learn_forget giữa 'train' và 'zero': "
          "chênh lệch đó chính là phần thay đổi quỹ đạo DO ĐỒ THỊ gây ra.")
else:
    print("Chưa ghi quỹ đạo nào (RECORD_TRAJECTORY=False).")


In [ ]:
# Vẽ lại hình đã lưu (heatmap + đường cong) cho từng tập dữ liệu và từng biến thể đồ thị
import matplotlib.image as mpimg

for name, run in result["runs"].items():
    pdir = Path(run["save_dir"]) / "trajectories"
    if not pdir.exists():
        continue
    for kind in sorted({p.stem.split("_")[-1] for p in pdir.glob("heatmap_*.png")}):
        png = pdir / f"heatmap_{kind}.png"
        if png.exists():
            print(f"[{name}] {png.name}")
            plt.figure(figsize=(9, 4))
            plt.imshow(mpimg.imread(png))
            plt.axis("off")
            plt.tight_layout()
            plt.show()

In [ ]:
# Một người học cụ thể: in bảng dài m(u,c,t) quanh các bước chuyển đổi quan trọng
name0 = list(result["runs"])[0]
run0 = result["runs"][name0]
mfile = Path(run0["save_dir"]) / "trajectories" / "mastery_train.csv"
if mfile.exists():
    mdf = pd.read_csv(mfile)
    print(mdf.head(12).to_string(index=False))
    piv = (mdf[mdf["answered"] == 1]
           .assign(d=lambda d: d["m_strong"] - d["m_pre"])
           .groupby("correct")["d"].agg(["mean", "count", "std"]))
    print("\nΔm tại khái niệm được trả lời, nhóm theo nhãn (phải dương khi đúng, âm khi sai):")
    display(piv)

## 6. Hiệu chỉnh dự đoán: ECE + Brier *(tuỳ chọn — mặc định bỏ qua vì `RUN_CALIBRATION=False`)*

Đề bài Cấp độ 2 chỉ yêu cầu **một** phân tích nâng cao và đã chọn quỹ đạo mastery (§5), nên phần này tắt mặc định. Đặt `RUN_CALIBRATION = True` để chạy khi session còn thời lượng.

Cùng một định nghĩa cho mọi mô hình (`level2/calibration.py`): `Brier = mean((p−y)²)`, `ECE = Σ_b (n_b/N)|avg(p_b) − avg(y_b)|` với 10 ngăn đều nhau. Dự đoán lấy trên toàn bộ vị trí có nhãn của tập test, dùng đúng mặt nạ mà AUC/ACC đã dùng.

Vì sao cần: AUC chỉ đo **thứ tự**, không đo độ tin cậy của con số xác suất — mà con số đó chính là thứ hiển thị cho người học. Một mô hình AUC cao hơn nhưng ECE cao hơn chưa chắc đã đáng tin hơn trong hệ thống thật.

In [ ]:
for name, run in result["runs"].items():
    cal = run.get("calibration")
    if cal is None or len(cal) == 0:
        print(f"[{name}] bỏ qua (RUN_CALIBRATION=False).")
        continue
    print("=" * 72)
    print(f"[{name}] BẢNG HIỆU CHỈNH (sắp theo ECE tăng dần)")
    print("=" * 72)
    display(cal)
    png = Path(run["save_dir"]) / "reliability.png"
    if png.exists():
        import matplotlib.image as mpimg
        plt.figure(figsize=(7, 5.6))
        plt.imshow(mpimg.imread(png))
        plt.axis("off")
        plt.tight_layout()
        plt.show()

## 7. Đánh giá đa seed *(tuỳ chọn — mặc định bỏ qua vì `RUN_MULTI_SEED=False`)*

Yêu cầu "trung bình + độ lệch chuẩn của ≥ 3 seed" đã được thoả ở Cấp độ 1 cho DKT/GKT/GKT-Zero **trên đúng split này** (số liệu dẫn lại trong `README.md`). Vì GRKT nặng hơn nhiều lần và mỗi seed là một lần train riêng, phần này tắt mặc định; đặt `RUN_MULTI_SEED = True` nếu muốn thêm phương sai riêng của GRKT.

Khi chạy, GRKT được lặp lại với `seeds = [42, 123, 456]` trên **cùng một split** (seed split luôn là 42): chỉ đổi khởi tạo trọng số và thứ tự batch. Nếu đổi cả split thì phương sai báo cáo sẽ trộn hai nguồn khác nhau, không còn là kiểm chứng ổn định.

In [ ]:
for name, run in result["runs"].items():
    if "multiseed" not in run:
        print(f"[{name}] RUN_MULTI_SEED=False — bỏ qua.")
        continue
    print("=" * 72)
    print(f"[{name}] ĐA SEED GRKT (split cố định seed={SPLIT_SEED})")
    print("=" * 72)
    display(run["multiseed_raw"])
    display(run["multiseed"])

## 8. Sản phẩm và cách đưa vào báo cáo

In [ ]:
from level2.dataset_spec import CANONICAL_COLUMNS

print("=== DANH SÁCH SẢN PHẨM CẤP ĐỘ 2 ===")
for name, run in result["runs"].items():
    print(f"\n[{name}] thư mục: {run['save_dir']}")
    for key, path in run["artifacts"].items():
        p = Path(path)
        if p.exists():
            size = p.stat().st_size
            print(f"  {key:22s} {p.name:34s} {size/1024:8.1f} KB")
        else:
            print(f"  {key:22s} {path}  (CHƯA TỒN TẠI)")
    ckpts = sorted(Path(run['save_dir']).glob('GRKT-*.pt'))
    print("  checkpoints           :", [c.name for c in ckpts])

print("\nBảng nối hai tập dữ liệu:", Path(SAVE_DIR) / "level2_all_datasets.csv")
print("Schema CSV chuẩn hoá (một dạng duy nhất cho mọi tập dữ liệu):", CANONICAL_COLUMNS)
print("Muốn tải tất cả về máy trong một lần: chạy cell đóng gói ở mục 8.5 bên dưới.")

## 8.5 Đóng gói toàn bộ sản phẩm để tải về

Kaggle xoá `/kaggle/working` khi session kết thúc, còn tab **Output** chỉ liệt kê từng file rời rạc. Cell dưới gộp cả cây `level2_out/` thành **một file zip** đặt ngay tại `/kaggle/working`, tải bằng nút tải xuống của tab Output (hoặc sidebar **Files**) **trước khi tắt session**.

Mặc định bỏ hai thứ không nên nằm trong zip:

* `junyi_canonical.csv` — CSV chuẩn hoá tái sinh được bằng `ensure_canonical()`, có thể hàng trăm MB.
* file nào lớn hơn `MAX_MB` — in danh sách kèm lý do để bạn tự quyết định, thay vì âm thầm cắt.

Checkpoint `GRKT-<graph>.pt` và `baselines/*.pt` **được giữ**: đó là duy nhất trong sản phẩm không dựng lại được khi hết session. Muốn zip nhẹ hơn nữa (chỉ để xem số liệu), đặt `KEEP_CKPTS = False`; muốn chắc chắn có hết, tăng `MAX_MB`.

In [ ]:
import zipfile

# ==== 3 núm điều chỉnh (chỉ sửa 3 thứ này nếu zip không như ý) ====
MAX_MB = 150                      # bỏ file lớn hơn mức này (tăng lên nếu muốn lấy hết)
KEEP_CKPTS = True                 # False = bỏ cả .pt/.npz, zip chỉ còn số liệu + ảnh
SKIP_NAMES = {"junyi_canonical.csv"}   # tái sinh được bằng ensure_canonical(), không cần tải

ZIP_PATH = Path("/kaggle/working/level2_output.zip") if IS_KAGGLE else (REPO / "level2_output.zip")

if not SAVE_DIR.exists():
    print(f"Không có thư mục output {SAVE_DIR} — hãy chạy cell thực nghiệm (§3) trước.")
else:
    zip_real = ZIP_PATH.resolve()
    added, skipped = [], []
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in sorted(SAVE_DIR.rglob("*")):
            if not p.is_file() or p.resolve() == zip_real:
                continue
            rel = str(p.relative_to(SAVE_DIR.parent))   # "level2_out/<dataset>/<file>"
            mb = p.stat().st_size / 1024 / 1024
            if p.name in SKIP_NAMES:
                skipped.append((rel, mb, "bỏ theo SKIP_NAMES (tái sinh được)"))
                continue
            if (not KEEP_CKPTS) and p.suffix in (".pt", ".npz"):
                skipped.append((rel, mb, "KEEP_CKPTS=False"))
                continue
            if mb > MAX_MB:
                skipped.append((rel, mb, f"lớn hơn MAX_MB={MAX_MB}"))
                continue
            zf.write(p, arcname=rel)
            added.append((rel, mb))

    print(f">>> {ZIP_PATH} | {ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB "
          f"(nén từ {len(added)} file, {sum(m for _, m in added):.1f} MB chưa nén)")
    bad = zipfile.ZipFile(ZIP_PATH).testzip()
    print("    kiểm tra toàn vẹn zip:", "OK" if bad is None else f"HỎNG tại {bad}")

    # gộp theo người dùng đọc được: cấp "level2_out/<dataset>" -> tên dataset, file nằm ngay
    # trong level2_out (level2_all_datasets.csv) -> giữ nguyên tên file
    groups: dict = {}
    for rel, mb in added:
        parts = rel.split("/")
        key = parts[1] if len(parts) > 1 else parts[0]
        n, tot = groups.get(key, (0, 0.0))
        groups[key] = (n + 1, tot + mb)
    print("    bên trong zip:")
    for key, (n, tot) in sorted(groups.items()):
        print(f"      {key:34s} {n:4d} file  {tot:9.1f} MB")

    if skipped:
        print(f"    bỏ {len(skipped)} file (tăng MAX_MB / sửa SKIP_NAMES / KEEP_CKPTS nếu muốn lấy cả):")
        for rel, mb, why in skipped:
            print(f"      {rel:52s} {mb:9.1f} MB | {why}")

    print("\n    Cách tải: tab Output của notebook Kaggle -> level2_output.zip -> biểu tượng tải xuống.")
    print("    Phải tải TRƯỚC khi session kết thúc (/kaggle/working bị xoá). Chạy lại cell này bất kỳ")
    print("    lúc nào để gói thêm sản phẩm — zip cũ bị ghi đè, không tạo file thừa.")

## 9. Sự cố thường gặp

| Hiện tượng | Nguyên nhân và cách xử lý |
|---|---|
| Chạy smoke mất hàng chục phút | `QUICK_RUN=True` nhưng `DATASETS` vẫn là `["assist2009"]` → đổi thành `["assist2009_smoke"]` (15 người học). |
| `ModuleNotFoundError: No module named 'level2'` (ngay tại cell import) | Repo trên GitHub chưa có code Cấp độ 2: notebook `git clone` từ `origin/main`, nên mọi sửa đổi phải được **commit và push** trước khi chạy. Kiểm tra: `!git -C /kaggle/working/Knowledge-Tracing log --oneline -1` và `!ls /kaggle/working/Knowledge-Tracing/level2` |
| `Không thấy file gốc của junyi` | Chưa Agree/Subscribe dataset, hoặc chưa `Add input`. Kiểm tra bằng `find_kaggle_input(patterns=("junyi",))` |
| Notebook chết giữa chừng khi chuẩn hoá Junyi (`Killed`, "Session crashed", hết RAM) | `Log_Problem.csv` có hàng chục triệu dòng. `_canonicalize_junyi` chỉ đọc các cột khớp `JUNYI.columns` (log in ra "đọc N/M cột" và số dòng đã nạp). Lưu ý `MAX_USERS` **không** giảm RAM lúc đọc vì nó lấy mẫu SAU khi đã nạp hết file. Nếu vẫn hết RAM: rút `JUNYI.columns` xuống đúng tên cột thật của file, hoặc chuẩn hoá ở notebook CPU riêng rồi attach CSV kết quả. |
| `json.decoder.JSONDecodeError` khi mở `outputs/level1/split.json` | File JSON trong repo là **con trỏ Git LFS** (xem `.gitattributes`), không phải nội dung. Code Cấp độ 2 cố tình không đọc nó: split được tái sinh bằng `load_kt_data(seed=42, train_ratio=0.6, val_ratio=0.2, min_seq_len=3, max_seq_len=200)` rồi đối chiếu từng người học bằng `_assert_align` |
| `AttributeError: 'Series' object has no attribute 'barh'` (hoặc `.hist`) khi vẽ EDA | Image Kaggle hiện chạy **pandas 3.x**, đã bỏ các accessor vẽ hình nối tiếp trên Series (`Series.barh`, `Series.hist`, `DataFrame.plot` dựng qua matplotlib backend cũ). Code trong repo đã đổi sang gọi matplotlib thuần (`axes[0].hist(...)`, `axes[1].barh(...)`); nếu gặp ở chỗ khác, sửa tương tự thay vì hạ cấp pandas. |
| `Cột ... không tồn tại` khi chuẩn hoá | Kaggle đổi tên cột. Mở `level2/dataset_spec.py`, thêm tên cột mới vào danh sách ứng viên trong `JUNYI.columns` (cell EDA đã in ra danh sách cột thật) |
| GRKT AUC ≈ 0.5 vài epoch đầu | Bình thường: GRKT khởi tạo ngẫu nhiên và có `gumbel_softmax` ở cả eval. Nếu epoch cuối vẫn ≈ 0.5, tăng `EPOCHS` hoặc giảm `LR` xuống 5e-4 |
| CUDA out of memory | Giảm `BATCH_SIZE` (64 → 32), `K_HIDDEN` (16 → 8), rồi mới tới `MAX_SEQ_LEN` (200 → 120; nhớ rằng khi đó mục 4 không còn đối chiếu khớp chính xác với Cấp độ 1). `RECORD_TRAJECTORY` ghi cả vector mastery cho mọi khái niệm: giảm `trajectory_users` |
| `KeyError: ... Split của Cấp độ 1 và dữ liệu đã nạp không cùng một lượt tiền xử lý` | `MAX_USERS`/`min_interactions`/`MAX_SEQ_LEN` đã đổi sau khi sinh CSV chuẩn hoá. Xoá `level2_out/junyi/junyi_canonical.csv` rồi chạy lại từ cell chuẩn hoá |
| `AssertionError: Chuỗi khái niệm của user ... khác Cấp độ 1` | Đây là **lỗi chủ đích**: nó chứng minh Cấp độ 2 đang nhìn dữ liệu khác Cấp độ 1. Dừng và kiểm tra lại `min_seq_len`/file CSV, không được tắt assert |
| `TypeError: only length-1 arrays can be converted to Python scalars` ở `[C2-B7]` (vẽ `reliability.png`) | Đã sửa trong `level2/calibration.py`: `markersize` phải là vô hướng. Nếu gặp lỗi vẽ hình tương tự sau này, các bước vẽ đã được bọc nên **số liệu CSV vẫn còn**; đặt `RESUME_GRKT_CKPT = True` rồi chạy lại cell thực nghiệm (§3) để đi tiếp từ checkpoint GRKT đã train, không tốn GPU train lại |
| Thiếu file trong `level2_output.zip` | Cell §8.5 in ra danh sách file bị bỏ kèm lý do: `SKIP_NAMES` (CSV chuẩn hoá tái sinh được) hoặc vượt `MAX_MB`. Tăng `MAX_MB` rồi chạy lại cell — zip được ghi đè. Nếu zip rỗng, cell đã chạy trước cell thực nghiệm. |
| Thời gian chạy vượt giới hạn Kaggle (9 giờ/tuần cho GPU) | Cấu hình mặc định đã tắt ablation/calibration/đa seed, nên mỗi tập chỉ còn MỘT lần train GRKT. Nếu vẫn vượt: chạy riêng từng tập (`DATASETS = ["junyi"]`, rồi `DATASETS = ["assist2009"]` ở session sau), giảm `EPOCHS` 5 → 3, hoặc chạy trước `QUICK_RUN=True` để chắc mã chạy hết chuỗi rồi mới chạy số liệu báo cáo. `RUN_BASELINES` gần như miễn phí vì nạp checkpoint Cấp độ 1 đã commit |
